In [0]:
# dbutils.fs.rm("/Volumes/swiggystreaming/raw/swiggyvolume/bronze/events", True)
# dbutils.fs.rm("/Volumes/swiggystreaming/raw/swiggyvolume/silver", True)
# dbutils.fs.rm("/Volumes/swiggystreaming/raw/swiggyvolume/gold", True)
# dbutils.fs.rm("/Volumes/swiggystreaming/raw/swiggyvolume/schema/events_schema", True)
# dbutils.fs.rm("/Volumes/swiggystreaming/raw/swiggyvolume/checkpoints/events_bronze", True)
# dbutils.fs.mkdirs("/Volumes/swiggystreaming/raw/swiggyvolume/schema/events_schema")
# dbutils.fs.mkdirs("/Volumes/swiggystreaming/raw/swiggyvolume/checkpoints/events_bronze")
# dbutils.fs.mkdirs("/Volumes/swiggystreaming/raw/swiggyvolume/checkpoints/test_bronze")
def run_bronze():
    df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/Volumes/swiggystreaming/raw/swiggyvolume/schema/events_schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("multiLine", "true")
        .load("/Volumes/swiggystreaming/raw/swiggyvolume/events/")
    )

    (df.writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/swiggystreaming/raw/swiggyvolume/checkpoints/events_bronze")
        .trigger(once=True)
        .outputMode("append")
        .start("/Volumes/swiggystreaming/raw/swiggyvolume/bronze/events")
        .awaitTermination()
    )


In [0]:
def run_silver():
    from pyspark.sql.functions import col

    bronze_df = spark.read.format("delta").load("/Volumes/swiggystreaming/raw/swiggyvolume/bronze/events")

    silver_orders = (
        bronze_df.filter("event_type = 'order'")
        .select("event_type","order_id","user_id","restaurant_id","total_amount","order_status","order_ts")
        .withColumn("order_ts", col("order_ts").cast("timestamp"))
    )

    silver_restaurant_status = (
        bronze_df.filter("event_type = 'restaurant_status'")
        .select("event_type","restaurant_id","is_open","prep_time_estimate","updated_ts")
        .withColumn("updated_ts", col("updated_ts").cast("timestamp"))
    )

    silver_delivery_status = (
        bronze_df.filter("event_type = 'delivery_status'")
        .select("event_type","order_id","delivery_partner_id","status",
                col("location.lat").alias("lat"),
                col("location.lon").alias("lon"),
                "updated_ts")
        .withColumn("updated_ts", col("updated_ts").cast("timestamp"))
    )

    silver_orders.write.format("delta").mode("overwrite").save("/Volumes/swiggystreaming/raw/swiggyvolume/silver/orders")
    silver_restaurant_status.write.format("delta").mode("overwrite").save("/Volumes/swiggystreaming/raw/swiggyvolume/silver/restaurant_status")
    silver_delivery_status.write.format("delta").mode("overwrite").save("/Volumes/swiggystreaming/raw/swiggyvolume/silver/delivery_status")

In [0]:
def run_gold():
    from pyspark.sql.functions import col, desc, row_number
    from pyspark.sql.window import Window

    silver_orders = spark.read.format("delta").load("/Volumes/swiggystreaming/raw/swiggyvolume/silver/orders")
    silver_restaurant = spark.read.format("delta").load("/Volumes/swiggystreaming/raw/swiggyvolume/silver/restaurant_status")
    silver_delivery = spark.read.format("delta").load("/Volumes/swiggystreaming/raw/swiggyvolume/silver/delivery_status")

    w_rest = Window.partitionBy("restaurant_id").orderBy(desc("updated_ts"))

    latest_restaurant_status = (
        silver_restaurant
        .withColumn("rn", row_number().over(w_rest))
        .filter("rn = 1")
        .drop("rn")
    )

    order_with_restaurant = (silver_orders.alias("o").join(
        latest_restaurant_status
            .drop("event_type")          # remove duplicate
            .drop("restaurant_id"),      # remove duplicate
        col("o.restaurant_id") == col("restaurant_id"),
        "left"
    )
)

    order_lifecycle = (
        order_with_restaurant.alias("o")
        .join(
            silver_delivery
                .withColumnRenamed("order_id", "delivery_order_id")
                .withColumnRenamed("updated_ts", "delivery_updated_ts")
                .withColumnRenamed("event_type", "delivery_event_type")
                .withColumnRenamed("status", "delivery_status"),
            col("o.order_id") == col("delivery_order_id"),
            "left"
        )
    )

    order_lifecycle.write.format("delta").mode("overwrite").save(
        "/Volumes/swiggystreaming/raw/swiggyvolume/gold/order_lifecycle"
    )

In [0]:
print("Starting Swiggy Streaming Pipeline...")

run_bronze()
run_silver()
run_gold()

print("Pipeline completed successfully.")

In [0]:
dbutils.fs.ls("/Volumes/swiggystreaming/raw/swiggyvolume/gold")